In [ ]:
import pickle
import json
import numpy as np
from scipy.signal import resample


# ============================================================
# SETTINGS
# ============================================================

WESAD_FILE = 'data/WESAD/S2.pkl'

OUTPUT_FILE = "test_window.json"

WINDOW_SIZE = 200

# Change this to select a different part of the subject
WINDOW_START = 0


# ============================================================
# LOAD WESAD
# ============================================================

print("Loading WESAD...")

with open(WESAD_FILE, "rb") as f:
    data = pickle.load(f, encoding="latin1")


# ============================================================
# GET WRIST SIGNALS
# ============================================================

wrist = data["signal"]["wrist"]

eda = wrist["EDA"].flatten()
bvp = wrist["BVP"].flatten()
temp = wrist["TEMP"].flatten()
acc = wrist["ACC"]


print("Original shapes:")
print("EDA :", eda.shape)
print("BVP :", bvp.shape)
print("TEMP:", temp.shape)
print("ACC :", acc.shape)


# ============================================================
# RESAMPLE EXACTLY LIKE load_subject()
# ============================================================

target_len = len(eda)

bvp = resample(
    bvp,
    target_len
)

acc_x = resample(
    acc[:, 0],
    target_len
)

acc_y = resample(
    acc[:, 1],
    target_len
)

acc_z = resample(
    acc[:, 2],
    target_len
)


# ============================================================
# COMBINE SIGNALS
# ============================================================

combined = np.column_stack([
    eda,
    bvp,
    temp,
    acc_x,
    acc_y,
    acc_z
])


print()
print("After resampling:")
print("Combined shape:", combined.shape)


# ============================================================
# EXTRACT WINDOW
# ============================================================

start = WINDOW_START
end = start + WINDOW_SIZE

window = combined[start:end]


# ============================================================
# CHECK WINDOW
# ============================================================

if len(window) != WINDOW_SIZE:

    raise ValueError(
        f"Window only contains {len(window)} samples. "
        f"Expected {WINDOW_SIZE}."
    )


# ============================================================
# CONVERT TO JSON-SAFE VALUES
# ============================================================

window = window.tolist()


# ============================================================
# CREATE API INPUT
# ============================================================

output = {
    "data": window
}


# ============================================================
# SAVE
# ============================================================

with open(OUTPUT_FILE, "w") as f:

    json.dump(
        output,
        f,
        indent=4
    )


# ============================================================
# DISPLAY INFORMATION
# ============================================================

print()
print("========================================")
print("WESAD WINDOW EXPORTED")
print("========================================")

print(f"Output file: {OUTPUT_FILE}")
print(f"Window start: {start}")
print(f"Window end: {end}")
print(f"Window shape: ({len(window)}, {len(window[0])})")

print()
print("Column order:")
print("0 = EDA")
print("1 = BVP")
print("2 = TEMP")
print("3 = ACC X")
print("4 = ACC Y")
print("5 = ACC Z")

print()
print("First row:")
print(window[0])

Loading WESAD...
ACC shape: (194528, 3)
BVP shape: (389056,)
EDA shape: (24316,)
TEMP shape: (24316,)


TypeError: only length-1 arrays can be converted to Python scalars